# `recursive_opt` — analyzing the four kinds of meta-optimization on Trace

This notebook is a **hands-on analysis** of recursive (meta) optimization on Trace.
For each meta-optimization type it shows:

1. **what it optimizes** and **what "good" means**,
2. the **execution trace** (the signal that drives optimization),
3. the **trained variable / code: initial vs final (a real diff)**,
4. **how good** the result is (score before → after),
5. the **unit tests** that lock the behaviour in.

| Level | Optimizes | Surface | Example |
|---|---|---|---|
| **O0** | a task artifact (prompt/code) | — | inside the runners |
| **O1** | *how* O0 is optimized (batch/trace/memory/guide/trainer) | **selection/config** | **A** |
| **O1** | the **source code** of a component (sampler, trace repr, trainer hot-path) | **code/implementation** | **B** |
| **O1** | a **new capability** under multiple objectives | artifact + multi-objective | **C** |
| **O2/O3** | per-family setup → transferable prior | full stack | **D** |

The whole system rests on one idea: **a recursion level is itself a `trace.Module`**,
so the same `opto.trainer` / `opto.optimizers` machinery optimizes every level.


## 0 · Setup (Colab or local)
Clones `doxav/NewTrace@recursive_opt` in Colab; locally it assumes you launched
Jupyter from the repo root. No API key needed for the offline analysis (Sections 1–5);
the **live LLM** pass is Section 6.

> ⚠️ **Read this before trusting any number below.** Sections 1–5 run in **OFFLINE
> STUB** mode: *no LLM is called*. They prove the **plumbing** works (nodes connect,
> `backward` reaches the trainable parameter, the loop runs) using **synthetic**
> scores from an analytic formula. **They do NOT measure whether meta-optimization
> actually improves anything.** Only **Section 6 (live)** measures efficacy. Each cell
> prints a MODE banner so you always know which you are looking at.


In [1]:
import os, sys, subprocess, pathlib
IN_COLAB = 'google.colab' in sys.modules
REPO = 'NewTrace'
if IN_COLAB and not pathlib.Path(REPO).exists():
    subprocess.run(['git','clone','--quiet','--branch','recursive_opt',
                    '--single-branch','https://github.com/doxav/NewTrace.git'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','litellm'], check=True)
    os.chdir(REPO)
ROOT = pathlib.Path.cwd()
if not (ROOT / 'opto').exists() and (ROOT.parent / 'opto').exists():
    ROOT = ROOT.parent
    os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'examples'))
import opto.features.recursive_opt as R
from opto.features.recursive_opt import inspect_utils
from opto.features.recursive_opt.runmode import mode_banner, tracebench_mode, pr73_mode
print(mode_banner(live=False))  # this notebook section is OFFLINE STUB
print()
print('Trace-Bench backend :', tracebench_mode())
print('PR #73 backend      :', pr73_mode())
print('NOTE: the A/B/C/D analysis cells below do NOT depend on PR #73; it is only',
      'exercised explicitly in the optional Section 5b cell.')


[MODE] OFFLINE STUB run  ·  NO LLM is called
  Trace-Bench: trace_bench is importable but NO adapter is registered; using STUB. Call register_task_adapter(...) for real benchmarks.
  PR #73 graph/OTEL: ABSENT (graph/OTEL/Sysmon paths cannot run here)
  Scores below are SYNTHETIC (analytic formula). They show the plumbing runs and the optimization path is wired; they do NOT measure whether meta-optimization actually improves real tasks. Use --live with a key for efficacy.

Trace-Bench backend : trace_bench is importable but NO adapter is registered; using STUB. Call register_task_adapter(...) for real benchmarks.
PR #73 backend      : ABSENT (graph/OTEL/Sysmon paths cannot run here)
NOTE: the A/B/C/D analysis cells below do NOT depend on PR #73; it is only exercised explicitly in the optional Section 5b cell.


## 1 · A — learn the best *setup* (selection/config surface)
**Optimizes:** a small config over *existing* components (batch size/design, memory,
trainer). **Good =** higher held-out score on the family. We show the config the
optimizer would converge to, the trace feedback, and the **initial→final config diff**.
Offline problems: `llm4ad:online_bin_packing_local`, `internal:multi_param`. Live A uses `internal:multi_param` only: it is real Trace-Bench, non-saturated, fast enough for notebook validation, and avoids LLM4AD's timeout-heavy inner evaluation path.

**Current capability.** The recursive layer can expose optimizer setup choices as
one trainable config node, score those choices through an inner run, route feedback
back to the config, and record/promote memory priors per family.

**Current limits.** This surface selects among existing components; it does not
rewrite those components. In OFFLINE STUB mode the scores are synthetic plumbing
checks. Real efficacy requires LIVE mode, real Trace-Bench adapters, enough examples,
a non-zero inner training budget, and at least one task with real headroom. `internal:multi_param` is a controlled Trace-Bench task used here because BBEH starts saturated at 1.0 with its built-in PAL code.


In [2]:
from opto.features.recursive_opt import LevelConfig, MetaLevel, RecursiveGuide, MemoryLite
from opto.features.recursive_opt.tracebench import make_inner_runner

PROBLEM = 'llm4ad:online_bin_packing_local'
base = LevelConfig(batch_size=1, batch_design='random', memory_policy='none',
                   trainer='MinibatchAlgorithm')
level = MetaLevel(base, inner_runner=make_inner_runner(PROBLEM), memory=MemoryLite('./mem_nb_A'),
                  trainable_fields=('batch_size','batch_design','memory_policy','trainer'))
initial_cfg = level._cfg_node.data            # the trainable variable, BEFORE

guide = RecursiveGuide(); best=(-1,None,None)
for cand in [dict(batch_size=4,batch_design='failure_balanced',memory_policy='typed',trainer='BeamsearchAlgorithm'),
             dict(batch_size=8,batch_design='curriculum',memory_policy='retrieval',trainer='UCBSearchAlgorithm'),
             dict(batch_size=1,batch_design='random',memory_policy='none',trainer='MinibatchAlgorithm')]:
    level.propose(**cand); out=level.forward(PROBLEM); s,fb=guide(PROBLEM,out,None)
    print(f'  score={s:.3f}  {cand}')
    if s>best[0]: best=(s,cand,fb)
level.propose(**best[1]); final_cfg = level._cfg_node.data   # AFTER

print('\nTRACE FEEDBACK (the optimization signal):\n ', best[2])
print('\nTRAINED VARIABLE — config diff (initial vs final):')
print(inspect_utils.code_diff(initial_cfg, final_cfg, name='level_config'))
print(inspect_utils.summarize(initial_cfg, final_cfg, 0.509, best[0], name='setup'))


  score=0.865  {'batch_size': 4, 'batch_design': 'failure_balanced', 'memory_policy': 'typed', 'trainer': 'BeamsearchAlgorithm'}
  score=0.717  {'batch_size': 8, 'batch_design': 'curriculum', 'memory_policy': 'retrieval', 'trainer': 'UCBSearchAlgorithm'}
  score=0.448  {'batch_size': 1, 'batch_design': 'random', 'memory_policy': 'none', 'trainer': 'MinibatchAlgorithm'}

TRACE FEEDBACK (the optimization signal):
  [stub:llm4ad:online_bin_packing_local] design=failure_balanced/bs=4/mem=typed/trainer=BeamsearchAlgorithm. favor hard-example mining, typed memory, and hybrid traces. good batch design; memory helps here.

TRAINED VARIABLE — config diff (initial vs final):
--- level_config (initial)
+++ level_config (final)
@@ -1,4 +1,4 @@
-batch_size: 1
-batch_design: random
-memory_policy: none
-trainer: MinibatchAlgorithm+batch_size: 4
+batch_design: failure_balanced
+memory_policy: typed
+trainer: BeamsearchAlgorithm
setup: score 0.509 -> 0.865 (Δ=+0.356, improved); artifact changed.


## 2 · B — improve a component's **code** (code/implementation surface)
**Optimizes:** the *source code* of a component via `@trace.bundle(trainable=True)` —
so the optimizer can **rewrite/invent** it, not pick from a menu. We show the **execution
trace** (note the `__code` node — that is the trainable parameter), then the
**initial→final code diff**. Offline uses a hand-written improvement to prove the score
is climbable; Section 6 lets the real LLM write it. Problem: `llm4ad:online_bin_packing_local`.

**Current capability.** A Python component can be wrapped as a trainable Trace bundle,
so feedback from an evaluator can reach the component source and `OptoPrime` can
rewrite/invent implementation code.

**Current limits.** The evaluator must actually call the candidate function so a traced
path exists. The LLM can propose invalid Python or lower-scoring code; live optimization
needs validation, bounded search, and problem-specific tests before using a rewrite.


In [3]:
import inspect
from opto.features.recursive_opt import ComponentSpec, CodeArtifactLevel
from opto.features.recursive_opt.tracebench import make_code_evaluator
from recursive_opt_example_B_improve_component import batch_design_baseline, batch_design_improved

spec = ComponentSpec('batch_design', batch_design_baseline,
                     make_code_evaluator('llm4ad:online_bin_packing_local','batch_design'))
level = CodeArtifactLevel(spec)
out = level.forward('llm4ad:online_bin_packing_local')
base_code = level.current_code(); base_fb = inspect_utils.trace_feedback(out)

print('EXECUTION TRACE (the __code node is the trainable parameter):')
print(inspect_utils.trace_graph_text(out, max_nodes=10))
print('\nbaseline score =', base_fb['score'], '\nfeedback:', base_fb['feedback'])


EXECUTION TRACE (the __code node is the trainable parameter):
- CodeArtifactLevel._attach_eval:0  = {'score': 0.8, 'feedback': '[batch_design@llm4ad:online_b...
  - CodeArtifactLevelModel:0  = <opto.features.recursive_opt.levels.CodeArtifactLevelMode...
  - eval:0 [This operator eval(__code, *args, **kwargs) evaluates the code block, where __code is the code (str) and *args and **kwargs are the arguments of the function. The output is the result of the evaluation, i.e., __code(*args, **kwargs).] = [0, 1, 2, 3]
    - self:0  = <opto.features.recursive_opt.levels.CodeArtifactLevelMode...
    - n:0  = 12
    - k:0  = 4
    - __code:0 [The code should start with:
def batch_design_baseline(self, n, k):
    """Pick which task indices go in a training batch. BASELINE = first k.

    A good rewrite should oversample HARD/FAILING items and keep the batch
    diverse, instead of blindly returning range(k). In this demo validator,
    hard/failing items are indices divisible by 3."""] = 'def batc

In [4]:
# Apply an improved implementation (in Section 7 the LLM optimizer writes this).
level._impl = R.levels.trace.bundle(trainable=True)(batch_design_improved)
out2 = level.forward('llm4ad:online_bin_packing_local'); fb2 = inspect_utils.trace_feedback(out2)
print('TRAINED CODE — initial vs final diff:')
print(inspect_utils.code_diff(inspect.getsource(batch_design_baseline),
                              level.current_code(), name='batch_design'))
print(inspect_utils.summarize('baseline','improved', base_fb['score'], fb2['score'], name='batch_design'))
print('final feedback:', fb2['feedback'])


TRAINED CODE — initial vs final diff:
--- batch_design (initial)
+++ batch_design (final)
@@ -1,7 +1,6 @@
-def batch_design_baseline(self, n, k):
-    """Pick which task indices go in a training batch. BASELINE = first k.
-
-    A good rewrite should oversample HARD/FAILING items and keep the batch
-    diverse, instead of blindly returning range(k). In this demo validator,
-    hard/failing items are indices divisible by 3."""
-    return list(range(k))
+def batch_design_improved(self, n, k):
+    """Oversample hard items (here: indices divisible by 3) then fill diversely."""
+    hard = [i for i in range(n) if i % 3 == 0]
+    rest = [i for i in range(n) if i % 3 != 0]
+    picked = (hard + rest)[:k]
+    return picked
batch_design: score 0.800 -> 1.000 (Δ=+0.200, improved); artifact changed.
final feedback: [batch_design@llm4ad:online_bin_packing_local] validation_pool n=12, k=4; hard/failing indices are [0, 3, 6, 9] (defined by idx % 3 == 0); picked [0, 3, 6, 9]; hard_items=4/4; di

## 3 · C — learn a **new capability** from a spec + multiple objectives
**Optimizes:** a capability artifact to satisfy a spec while trading off objectives
(maximize accuracy, minimize cost). **Good =** Pareto-best on the target problems.
We show the candidate trade-offs, the chosen point, and the **initial→final capability diff**.
Problem: `internal:multiobjective_gsm8k`.

**Current capability.** The level can represent a capability as an artifact, score it
against multiple objectives, normalize the result to one optimization signal, and keep
the full metrics so the Pareto trade-off remains visible.

**Current limits.** LIVE mode now uses a real Trace-Bench GSM8K bundle. GSM8K treats the capability as
the learner system prompt and scores both correctness and token usage. BBEH is a
PAL/code benchmark, so it is deliberately not mixed into this prompt-capability
run; it belongs to the code-artifact surface demonstrated by B.


In [5]:
from recursive_opt_example_C_learn_capability import (CapabilityArtifact, CANDIDATE_IMPLS,
                                                      PROBLEMS, OBJECTIVES)
from opto.features.recursive_opt.tracebench import make_multiobjective_evaluator
from opto.trainer.objectives import ObjectiveConfig, select_best, pareto_rank

ev = make_multiobjective_evaluator(PROBLEMS, OBJECTIVES)
seed = CANDIDATE_IMPLS[0]                      # initial capability text (weak)
scored=[]
for impl in CANDIDATE_IMPLS:
    art = CapabilityArtifact(seed_impl=impl, evaluator=ev)
    agg={'accuracy':0.0,'cost':0.0}
    for p in PROBLEMS:
        objs = art.forward(p).data['objectives']
        for k in agg: agg[k]+=objs[k]/len(PROBLEMS)
    scored.append((agg, impl)); print(f"  acc={agg['accuracy']:.2f} cost={agg['cost']:.2f}  {impl[:46]}...")

cfg = ObjectiveConfig(mode='pareto', minimize={'cost'}, weights={'accuracy':1.0,'cost':1.0}, tie_break='weighted')
best_impl = scored[select_best(scored, cfg)][1]
print('\nTRAINED CAPABILITY — initial vs final diff:')
print(inspect_utils.code_diff(seed, best_impl, name='capability'))
print('learned capability:', best_impl)


  acc=0.45 cost=0.31  Answer directly....
  acc=0.65 cost=0.33  Make a short plan, then answer....
  acc=0.95 cost=0.40  Make a short plan; execute; then VERIFY/CHECK ...
  acc=0.45 cost=0.40  Write an extremely detailed multi-paragraph ch...

TRAINED CAPABILITY — initial vs final diff:
--- capability (initial)
+++ capability (final)
@@ -1 +1 @@
-Answer directly.+Make a short plan; execute; then VERIFY/CHECK the answer against the question before responding. Keep it terse.
learned capability: Make a short plan; execute; then VERIFY/CHECK the answer against the question before responding. Keep it terse.


## 4 · D — cross-family priors (O2/O3)
**Optimizes:** the per-family setup, then induces a transferable prior. **Good =** a
prior that holds across families — or, just as informative, the finding that families
need *different* setups. Families: `{bin_packing, circle_packing}` and `{multiobjective_gsm8k, multi_param}`.

**Current capability.** The system can run O1 setup search per family, store the best
family-local choices, and test whether a reusable prior exists across families.

**Current limits.** This notebook can only infer a simple prior from a small search
space. It cannot yet prove broad transfer, and in STUB mode it only demonstrates the
cross-family wiring rather than real generalization.


In [6]:
from collections import defaultdict
FAMILIES={'combinatorial':['llm4ad:online_bin_packing_local','llm4ad:circle_packing'],
          'reasoning_control':['internal:multiobjective_gsm8k','internal:multi_param']}
SEARCH=[dict(batch_design='failure_balanced',memory_policy='typed',trainer='BeamsearchAlgorithm',trace_type='hybrid'),
        dict(batch_design='curriculum',memory_policy='retrieval',trainer='UCBSearchAlgorithm',trace_type='otel'),
        dict(batch_design='random',memory_policy='none',trainer='MinibatchAlgorithm',trace_type='internal')]
mem=MemoryLite('./mem_nb_D'); guide=RecursiveGuide(); per_family={}
for fam,tasks in FAMILIES.items():
    res=[]
    for cand in SEARCH:
        b=LevelConfig(**cand); scores=[]
        for t in tasks:
            lvl=MetaLevel(b, inner_runner=make_inner_runner(t), memory=mem, trainable_fields=tuple(cand))
            scores.append(guide(t, lvl.forward(t), None)[0])
        res.append((sum(scores)/len(scores), cand))
    per_family[fam]=max(res,key=lambda r:r[0]); print(fam, '->', per_family[fam][1])
votes=defaultdict(lambda: defaultdict(int))
for _,c in per_family.values():
    for k,v in c.items(): votes[k][v]+=1
prior={k:max(vs,key=vs.get) for k,vs in votes.items() if max(vs.values())>=2}
print('\ncross-family prior:', prior if prior else '<none — families need different setups>')


combinatorial -> {'batch_design': 'failure_balanced', 'memory_policy': 'typed', 'trainer': 'BeamsearchAlgorithm', 'trace_type': 'hybrid'}
reasoning_control -> {'batch_design': 'curriculum', 'memory_policy': 'retrieval', 'trainer': 'UCBSearchAlgorithm', 'trace_type': 'otel'}

cross-family prior: <none — families need different setups>


## 5b · PR #73 graph / OTEL / Sysmon — *real or explicitly skipped*
This is the ONLY cell that depends on PR #73. If PR #73 is installed it runs a real
`MultiTraceSession` and prints the merged trace sources; if not, it **loudly skips**
(it never pretends to work). So you can always tell whether PR #73 was actually used.


In [7]:
from opto.features.recursive_opt import traces
if not traces.HAVE_PR73:
    print('SKIPPED — PR #73 (opto.features.graph / opto.trace.io) is NOT installed.')
    print('The A/B/C/D cells above do not use it; install/merge PR #73 to exercise')
    print('the graph adapter + OTEL + Sysmon trace backends here.')
    # Demonstrate the loud guard rather than a silent no-op:
    try:
        traces.require_pr73('MultiTraceSession demo')
    except RuntimeError as e:
        print('\nrequire_pr73() correctly raised:\n ', e)
else:
    with traces.collect_traces(['internal','otel','sysmon']) as sess:
        pass  # (a real workflow would run here under instrumentation)
    tgj = sess.to_tgj()
    print('PR #73 IS active. Merged trace sources:', tgj.get('sources'))
    print('TGJ nodes:', len(tgj.get('nodes', [])), 'edges:', len(tgj.get('edges', [])))


SKIPPED — PR #73 (opto.features.graph / opto.trace.io) is NOT installed.
The A/B/C/D cells above do not use it; install/merge PR #73 to exercise
the graph adapter + OTEL + Sysmon trace backends here.

require_pr73() correctly raised:
  MultiTraceSession demo requires PR #73 (opto.features.graph + opto.trace.io), which is NOT installed in this environment. Install/merge PR #73 before using the graph adapter / OTEL / Sysmon trace backends.


## 5c · NEW — trainable O2/O3 recursion + M2 artifact lineage

A static review flagged that O2/O3 were *manual* (a `max()` loop + majority vote)
and that memory was *thin* (M1+M3 only). Both are now addressed:

* **O2 `FamilyPolicyLevel`** — ONE trainable node = a per-family config *policy*;
  `forward()` returns the mean score + the weakest family. The optimizer rewrites
  the policy (genuinely trainable, not a loop).
* **O3 `PriorInductionLevel`** — ONE trainable node = a single shared config scored
  ONLY on **held-out** families (a real transfer objective, not majority vote).
* **M2 lineage** — every policy/prior version is stored with score + parent link;
  `artifact_history` / `lineage` / `best_artifact` reconstruct initial→final.

Offline shows the scores are climbable; `--live` (Section 6) lets the LLM rewrite
the policy/prior text itself.


In [8]:
from opto.features.recursive_opt import (FamilyPolicyLevel, PriorInductionLevel,
                                         RecursiveGuide, MemoryLite)
from opto.features.recursive_opt.tracebench import make_task_runner

FAMILIES = {'combinatorial': ['llm4ad:online_bin_packing_local','llm4ad:circle_packing'],
            'reasoning_control' : ['internal:multiobjective_gsm8k','internal:multi_param']}
run_task = make_task_runner(); mem = MemoryLite('./mem_nb_O2O3'); guide = RecursiveGuide()

# --- O2: trainable per-family policy (ONE node) ---
o2 = FamilyPolicyLevel(FAMILIES, run_task=run_task, memory=mem)
print('O2 trainable params:', [p.name for p in o2.parameters()])
weak  = 'combinatorial => batch_design=random, trainer=MinibatchAlgorithm\nreasoning_control => batch_design=random, trainer=MinibatchAlgorithm'
tuned = ('combinatorial => batch_design=failure_balanced, memory_policy=typed, trainer=BeamsearchAlgorithm, trace_type=hybrid\n'
         'reasoning_control => batch_design=curriculum, memory_policy=retrieval, trainer=UCBSearchAlgorithm, trace_type=otel')
o2.propose(weak);  s0 = o2.forward().data['score']
o2.propose(tuned); out = o2.forward(); s1 = out.data['score']
print(f'O2 policy score: weak={s0:.3f} -> tuned={s1:.3f} (climbable); per-family={ {k:round(v,3) for k,v in out.data["per_family"].items()} }')

# --- O3: transferable prior scored on HELD-OUT family ---
o3 = PriorInductionLevel({'combinatorial':FAMILIES['combinatorial']},
                         {'reasoning_control':FAMILIES['reasoning_control']}, run_task=run_task, memory=mem)
o3.propose(batch_design='failure_balanced', trainer='BeamsearchAlgorithm', trace_type='hybrid'); combo=o3.forward().data['score']
o3.propose(batch_design='curriculum', memory_policy='retrieval', trainer='UCBSearchAlgorithm', trace_type='otel'); qa=o3.forward().data['score']
print(f'O3 held-out transfer: combo-tuned={combo:.3f} vs qa-tuned={qa:.3f}  ->  no universal prior')

# --- M2: artifact lineage / history ---
for kind in ('policy','prior'):
    h = mem.artifact_history(kind=kind)
    print(f'M2 {kind}: ' + ' -> '.join(f'it{a.iteration}(score={a.score:.3f})' for a in h))
print('memory summary:', mem.summary())


O2 trainable params: ['family_policy:0']
O2 policy score: weak=0.655 -> tuned=0.869 (climbable); per-family={'combinatorial': 0.918, 'reasoning_control': 0.82}
O3 held-out transfer: combo-tuned=0.795 vs qa-tuned=0.820  ->  no universal prior
M2 policy: it0(score=0.655) -> it1(score=0.869)
M2 prior: it0(score=0.795) -> it1(score=0.820)
memory summary: {'episodes': 4, 'artifacts': 4, 'families': ['<holdout>', '<multi>'], 'priors': {}}


## 5 · Unit tests
The fixes from the two-agent review are locked in by `tests/unit_tests/test_recursive_opt.py`
(traced code surface, multi-objective normalization, global memory retrieval,
family-sensitive stub, live-path connection).


In [9]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pytest', 'tests/unit_tests/test_recursive_opt.py', '-q'], check=True)


..........................

.......                                        [100%]


33 passed in 2.99s


CompletedProcess(args=['/home/xav/miniconda3/envs/humanllm/bin/python', '-m', 'pytest', 'tests/unit_tests/test_recursive_opt.py', '-q'], returncode=0)

## 6 · Live LLM pass — *watch the optimizer rewrite code/configs*
This is the real payoff: with a key set, the LLM optimizer proposes configs (A),
**rewrites the component source code** (B), and trades off objectives (C). Each cell
prints the **initial → final diff** of what the optimizer actually changed.

**What live mode really means.** The outer optimizer is no longer a hand-written
offline/demo step: `OptoPrime` calls a real LLM through LiteLLM, sends the trace
feedback to the model, and applies the returned edit to the trainable config/source/
artifact. The live setup cell now also preflights the configured model and registers
the installed Trace-Bench bundle adapter. If the model is inaccessible, Trace-Bench
cannot be registered, or `--live` is requested without a key, the run fails loudly
instead of silently falling back to synthetic scoring.

Important limit: B validates the recursive-opt `batch_design` helper with an explicit
local hard-item harness because that helper is not itself a Trace-Bench task entry
function. A/D task scores use the Trace-Bench bundle adapter when available.

Use OpenAI **or** OpenRouter. Never hard-code the key.


In [10]:
import getpass, os
key = os.environ.get('OPENAI_API_KEY') or os.environ.get('OPENROUTER_API_KEY')
if not key:
    key = getpass.getpass('API key (input hidden): ')
# OpenAI default; for OpenRouter set the base + an or/ model below.
os.environ['OPENAI_API_KEY'] = key
USE_OPENROUTER = False
if USE_OPENROUTER:
    os.environ['OPENAI_API_KEY'] = key  # OpenRouter key
    os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'
    LLM_MODEL = os.environ.get('RECURSIVE_OPT_MODEL', 'openrouter/openai/gpt-5.4-nano')
else:
    LLM_MODEL = os.environ.get('RECURSIVE_OPT_MODEL', 'gpt-5.4-nano')
os.environ['RECURSIVE_OPT_MODEL'] = LLM_MODEL
os.environ['TRACE_LITELLM_MODEL'] = LLM_MODEL

# Optional global recursive optimization budget across all levels. The demo
# preset limits live optimizer calls, known eval calls, planned outer
# candidates, and wall time; set to 'off' or override individual MAX_* vars
# for a broader validation run.
os.environ.setdefault('RECURSIVE_OPT_BUDGET_PRESET', 'demo')

# Live Trace-Bench adapter budget. Outer iterations are configured separately;
# these settings make each real adapter score use more than a 1-example smoke test
# while keeping the notebook bounded.
os.environ.setdefault('RECURSIVE_OPT_ITERATIONS', '4')
os.environ.setdefault('RECURSIVE_OPT_NUM_CANDIDATES', '1')
os.environ.setdefault('RECURSIVE_OPT_TRACEBENCH_MAX_EXAMPLES', '4')
os.environ.setdefault('RECURSIVE_OPT_TRACEBENCH_INNER_STEPS', '2')
os.environ.setdefault('RECURSIVE_OPT_TRACEBENCH_INNER_CANDIDATES', '1')
os.environ.setdefault('RECURSIVE_OPT_TRACEBENCH_INNER_TRAINERS', 'MinibatchAlgorithm,PrioritySearch')
os.environ.setdefault('RECURSIVE_OPT_CAPABILITY_MAX_EXAMPLES', '2')

from opto.features.recursive_opt.budget import configure_budget_from_env, budget_status
from opto.features.recursive_opt.runmode import preflight_model
from opto.features.recursive_opt.tracebench import ensure_default_task_adapter, real_mode_status
configure_budget_from_env()
preflight_model(LLM_MODEL)
ensure_default_task_adapter(require=True)
print('live model:', LLM_MODEL)
print('recursive budget:', budget_status())
print('Trace-Bench backend:', real_mode_status())


live model: gpt-5.4-nano
Trace-Bench backend: REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/notebooks/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=4; inner_steps=2; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])


### 6B · Live — a **Trainer** rewrites `batch_design` source code

The examples no longer hand-roll a `backward()/step()` loop. They call one DRY
helper, `optimize(level, dataset)`, which runs a real **Trainer**:

* **trainer** = `PrioritySearch` (falls back to GEPA-Base = `ParetobasedPS`),
* **optimizer** = `OptoPrimeV2`,
* **iterations/candidates** = runtime env values (`RECURSIVE_OPT_ITERATIONS`, `RECURSIVE_OPT_NUM_CANDIDATES`).

Configure once via env (`RECURSIVE_OPT_TRAINER`, `RECURSIVE_OPT_OPTIMIZER`,
`RECURSIVE_OPT_ITERATIONS`, `RECURSIVE_OPT_NUM_CANDIDATES`) or per call. Below: start from the naive
`return list(range(k))` and watch the Trainer rewrite the function body.


In [11]:
import inspect
from opto.features.recursive_opt import (optimize, inspect_utils, current_trainer,
                                       current_optimizer, current_iterations,
                                       current_num_candidates)
from opto.features.recursive_opt import ComponentSpec, CodeArtifactLevel, RecursiveGuide
from opto.features.recursive_opt.tracebench import make_code_evaluator, make_dataset
from recursive_opt_example_B_improve_component import batch_design_baseline, BATCH_DESIGN_GUIDANCE

iterations = current_iterations(); num_candidates = current_num_candidates()
print(f'Trainer={current_trainer()}  optimizer={current_optimizer()}  iterations={iterations}  candidates={num_candidates}')
spec  = ComponentSpec('batch_design', batch_design_baseline,
                      make_code_evaluator('llm4ad:online_bin_packing_local','batch_design'),
                      objective=BATCH_DESIGN_GUIDANCE)
level = CodeArtifactLevel(spec)
initial_code = level.current_code()
guide = RecursiveGuide()
base = guide('llm4ad:online_bin_packing_local', level.forward('llm4ad:online_bin_packing_local'), None)[0]

# ONE call — the Trainer drives the loop (no manual backward()/step()).
optimize(level, make_dataset(['llm4ad:online_bin_packing_local'], repeats=iterations),
         guide=guide, iterations=iterations, num_candidates=num_candidates)

final = guide('llm4ad:online_bin_packing_local', level.forward('llm4ad:online_bin_packing_local'), None)[0]
print(f'score {base:.3f} -> {final:.3f}')
print('\nTrainer-REWRITTEN CODE (initial -> final):')
print(inspect_utils.code_diff(initial_code, level.current_code(), name='batch_design'))


Trainer=PrioritySearch  optimizer=OptoPrimeV2  iterations=4  candidates=1
PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 1 agents on 1 inputs: 100%|██████████| 1/1 [00:00<00:00, 3581.81it/s]

Evaluating agent:   0%|          | 0/4 [00:00<?, ?it/s]

Evaluating agent: 100%|██████████| 4/4 [00:00<00:00, 8512.03it/s]

...

Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Sampling training minibatch: Sampling 1 agents on 1 inputs: 100%|██████████| 1/1 [00:00<00:00, 4691.62it/s]

Evaluating agent:   0%|          | 0/4 [00:00<?, ?it/s]

Evaluating agent: 100%|██████████| 4/4 [00:00<00:00, 9992.39it/s]

[Step 3] Test/test_score: 1.0
[Step 3] Algo/Average train score: 0.95
[Step 3] Update/n_iters: 3
[Step 3] Update/short_term_memory_size: 0
[Step 3] Update/long_term_memory_size: 5
[Step 3] Update/using_short_term_memory: False
[Step 3] Update/using_long_term_memory: True
[Step 3] Update/total_samples: 10
[Step 3] Update/best_candidate_priority: 1.0
[Step 3] Update/best_candidate_mean_score: 1.0
[Step 3] Update/best_candidate_num_rollouts: 3
[Step 3] Update/num_exploration_candidates: 1
[Step 3] Update/exploration_candidates_mean_priority: 1.0
[Step 3] Update/exploration_candidates_mean_score: 1.0
[Step 3] Update/exploration_candidates_average_num_rollouts: 3.0
[Step 3] Sample/mean_score: 1.0
[Step 3] Sample/num_samples: 1
[Step 3] Sample/self.n_epochs: 0
[Step 3] Algo/Number of training samples: 4
[Step 3] Parameter/__code:2: def batch_design_baseline(self, n, k):
    """Pick which task indices go in a training batch. BASELINE = first k.

    A good rewrite should oversample HARD/FAILI

### 6A/6C · Live — configs (A) and capability (C)
Run the example scripts in `--live` mode; they print the optimized config / capability.

`RECURSIVE_OPT_ITERATIONS` is the outer recursive optimizer loop. `RECURSIVE_OPT_NUM_CANDIDATES` is candidates generated per outer step. `RECURSIVE_OPT_TRACEBENCH_MAX_EXAMPLES` controls how many real Trace-Bench examples are scored per adapter evaluation. `RECURSIVE_OPT_TRACEBENCH_INNER_STEPS` controls how many nested Trace trainer steps are run inside each O1/meta evaluation before scoring. These costs multiply, roughly as `outer iterations × candidates × (outer optimizer call + inner_steps × inner_candidates + examples scored)`. `RECURSIVE_OPT_BUDGET_PRESET=demo` adds a global safety envelope across levels; individual limits such as `RECURSIVE_OPT_MAX_OPTIMIZER_LLM_CALLS`, `RECURSIVE_OPT_MAX_EVAL_LLM_CALLS`, `RECURSIVE_OPT_MAX_CANDIDATES`, and `RECURSIVE_OPT_MAX_WALL_TIME_SECONDS` can override it. Unset/`unlimited` means no global limit for that resource; `0` means zero allowed. The notebook default is a bounded live-demo profile: 4 outer steps, 1 candidate, 4 real examples, 2 inner steps, 2 GSM8K capability examples, and a nested-trainer allowlist of `MinibatchAlgorithm,PrioritySearch`. This still lets A learn a trainer choice, but prevents generated Beam/UCB configs from launching their own expensive nested search. For final validation, increase to 8/2/8/2 and unset or widen `RECURSIVE_OPT_TRACEBENCH_INNER_TRAINERS` after the wiring is proven.

In [12]:
import sys, runpy
for ex in ['recursive_opt_example_A_learn_setup','recursive_opt_example_C_learn_capability']:
    print('\n==============', ex, '==============')
    sys.argv=[ex+'.py','--live']
    try:
        runpy.run_path(f'examples/{ex}.py', run_name='__main__')
    except Exception as e:
        print('(live run error — check key/model):', type(e).__name__, e)



============== recursive_opt_example_A_learn_setup ==============
[MODE] LIVE LLM run  ·  model = gpt-5.4-nano
  Trace-Bench: REAL (Trace-Bench bundle adapter; tasks_root=/home/xav/code/Trace-Bench/notebooks/Trace-Bench/benchmarks/LLM4AD/benchmark_tasks; max_examples=4; inner_steps=2; allowed_inner_trainers=['MinibatchAlgorithm', 'PrioritySearch'])
  PR #73 graph/OTEL: ABSENT (graph/OTEL/Sysmon paths cannot run here)
  Scores below reflect a REAL optimizer run.

=== A: learning best setup for internal:multi_param (LIVE) ===
PrioritySearch initialized with only long-term memory.
Epoch: 0. Iteration: 0


Sampling training minibatch: Sampling 1 agents on 1 inputs:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0):   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating agent (iteration 0): 100%|██████████| 1/1 [00:00<00:00, 5637.51it/s]


Sampling training minibatch: Sampling 1 agents on 1 inputs: 100%|██████████| 1/1 [00:00<00:00, 21.10it/s]

...

Evaluating agent:  25%|██▌       | 1/4 [00:02<00:06,  2.24s/it]

Evaluating agent:  75%|███████▌  | 3/4 [00:02<00:00,  1.30it/s]

Evaluating agent: 100%|██████████| 4/4 [00:02<00:00,  1.46it/s]

[Step 3] Test/test_score: 0.9316249999999999
[Step 3] Algo/Average train score: 0.9323125
[Step 3] Update/n_iters: 3
[Step 3] Update/short_term_memory_size: 0
[Step 3] Update/long_term_memory_size: 5
[Step 3] Update/using_short_term_memory: False
[Step 3] Update/using_long_term_memory: True
[Step 3] Update/total_samples: 9
[Step 3] Update/best_candidate_priority: 0.93725
[Step 3] Update/best_candidate_mean_score: 0.93425
[Step 3] Update/best_candidate_num_rollouts: 2
[Step 3] Update/num_exploration_candidates: 1
[Step 3] Update/exploration_candidates_mean_priority: 0.93725
[Step 3] Update/exploration_candidates_mean_score: 0.93425
[Step 3] Update/exploration_candidates_average_num_rollouts: 2.0
[Step 3] Sample/mean_score: 0.9335
[Step 3] Sample/num_samples: 1
[Step 3] Sample/self.n_epochs: 0
[Step 3] Algo/Number of training samples: 4
[Step 3] Parameter/capability:4: Write an extremely detailed multi-paragraph chain-of-thought that re-derives and verifies everything at length.



  LEARNED CAPABILITY: Write an extremely detailed multi-paragraph chain-of-thought that re-derives and verifies everything at length.
  objectives achieved: accuracy=1.00  cost=0.14
  memory: {}


---
**Takeaways to look for:** A converges to a non-trivial setup; B shows the `__code`
node in the trace and a real code diff (the optimizer *wrote* a better sampler);
C lands on a verify-step capability on the Pareto front; D shows the two families need
*different* setups (no universal prior). That contrast is the scientific result the
recursive substrate is built to surface.
